# Nested Cross-Validation

Compares the Mean baseline, Linear, Lasso and Ridge models for each mental-health target
using nested cross-validation on the training set (the test set is not used here).
Lasso and Ridge tune alpha in an inner 5-fold CV; performance is measured on
5-fold × 10-repeat outer CV (50 folds).

In [ ]:
import pandas as pd

import mxmh_common as mc

# Set RERUN = True to re-run the full nested CV (about 11 minutes).
# Otherwise the saved per-fold results are loaded if they exist.
RERUN = True
FOLDS_PATH = mc.RESULTS_DIR / "nested_cv_folds.csv"

if RERUN or not FOLDS_PATH.exists():
    folds = mc.run_nested_cv()
else:
    folds = pd.read_csv(FOLDS_PATH)
    print(f"Loaded {len(folds)} fold results from {FOLDS_PATH}")

# Training and held-out sizes of each outer fold, needed for the corrected CI
X, Y = mc.load_clean_dataset()
X_train, _, _, _ = mc.split_train_test(X, Y)
n_train, n_test = mc.outer_fold_sizes(len(X_train))

## Performance Table

Mean across the 50 outer folds, with the corrected 95% CI (Nadeau & Bengio, 2003) for R², RMSE and MAE.

In [ ]:
MODEL_ORDER = list(mc.make_models().keys())


def fmt_ci(ci, digits):
    return f"{ci['mean']:.{digits}f} [{ci['ci_low']:.{digits}f}, {ci['ci_high']:.{digits}f}]"


rows = []
for (target, model), group in folds.groupby(["target", "model"]):
    rows.append({"Target": target, "Model": model,
                 "R2": fmt_ci(mc.corrected_ci(group.R2, n_train, n_test), 3),
                 "RMSE": fmt_ci(mc.corrected_ci(group.RMSE, n_train, n_test), 2),
                 "MAE": fmt_ci(mc.corrected_ci(group.MAE, n_train, n_test), 2)})
perf = pd.DataFrame(rows)
perf["Model"] = pd.Categorical(perf.Model, MODEL_ORDER, ordered=True)
perf = perf.sort_values(["Target", "Model"])
perf.to_csv(mc.RESULTS_DIR / "nested_cv_performance.csv", index=False)
perf.set_index(["Target", "Model"])